# SIMULACIÓN ENTRADA SÍMBOLO PARA REVERSIONES FUTUROS - BINANCE

In [1]:
# =========================================================
# LIBRERÍAS
# =========================================================
import requests
import os
import pandas as pd
import numpy as np
import ta
import json
from pathlib import Path

import time
from datetime import datetime, timezone, timedelta
import pytz

import winsound

import sys

In [2]:
# =========================================================
# CONFIGURACIÓN GLOBAL
# =========================================================
BINANCE_FUTURES_BASE = "https://fapi.binance.com"
INTERVAL = "15m"                 # coherente con scanner 1H
LIMIT = 120                     # ~15 días de contexto
LOOKBACK = 120
MAX_SYMBOLS_TRAIN = 540       # limitar universo de entrenamiento
MIN_NOTIONAL_VOLUME = 10_000  # Sugerido 5_000_000

AVG_LIMIT_LONG = 30    # límite de KPI AVG para reversión LONG 
AVG_LIMIT_SHORT = 70    # límite de KPI AVG para reversión SHORT 

FILEPATH = "C:/Users/Lenovo S540/Python_scripts/ListaSimbolosReversionesFuturos.txt"

In [3]:
# =========================================================
# MANEJO DE LISTA DE SÍMBOLOS
# =========================================================
def load_symbol_map(filepath: str) -> dict:
    """
    Carga archivo JSON con formato:
    {
        "ETHUSDT": "LONG",
        "LTCUSDT": "SHORT"
    }
    """
    try:
        path = Path(filepath)

        if not path.exists():
            return {}

        with open(path, "r") as f:
            data = json.load(f)

        # Validación básica
        valid = {}
        for k, v in data.items():
            if v in ["LONG", "SHORT"]:
                valid[k] = v

        return valid

    except Exception as e:
        print(f"[WARN] Error cargando JSON: {e}")
        return {}


def save_symbol_map(filepath: str, symbol_map: dict):
    """
    Guarda el diccionario actualizado en formato JSON.
    """
    try:
        with open(filepath, "w") as f:
            json.dump(symbol_map, f, indent=4)
    except Exception as e:
        print(f"[WARN] Error guardando JSON: {e}")


In [4]:
# =========================================================
# ENVÍO DE ALERTAS
# =========================================================
TELEGRAM_BOT_TOKEN = "8593522900:AAFSgAMZAKolZaYqm2GjeqY4Hr3tP7Jnk2M"
TELEGRAM_CHAT_ID = "1733579121"

def send_telegram_alert(message: str):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "Markdown"
    }
    try:
        requests.post(url, json=payload, timeout=5)
    except Exception as e:
        print(f"[WARN] Telegram alert failed: {e}")



def local_warning():

    frequency = 2500  # Set Frequency To 2500 Hertz
    duration = 1000   # Set Duration To 1000 ms == 1 second

    winsound.Beep(frequency, duration)
    print("Oportunidades de entrada encontradas!")
    
    # Example of beeping with time delays
    for i in range(3):
        winsound.Beep(500, 200) # Frequency 500 Hz, duration 200ms
        time.sleep(0.5)        # Pause for 0.5 seconds between beeps


In [5]:
# =========================================================
# 2️⃣ OHLC
# =========================================================
def get_ohlc(
    symbol: str,
    interval: str = "1h",
    limit: int = 500,
    max_total: int | None = None
):
    """
    Descarga OHLCV desde Binance Futures (USDT-M).
    
    - Usa requests (sin API key)
    - Soporta paginación si max_total > limit
    - Devuelve DataFrame limpio y ordenado
    """

    url = f"{BINANCE_FUTURES_BASE}/fapi/v1/klines"
    all_data = []
    end_time = None

    max_total = max_total or limit

    while len(all_data) < max_total:
        fetch = min(1500, max_total - len(all_data))

        params = {
            "symbol": symbol,
            "interval": interval,
            "limit": fetch
        }

        if end_time is not None:
            params["endTime"] = end_time

        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()
            
            if not data:
                break

            all_data = data + all_data
            end_time = data[0][0] - 1  # ms, vela anterior

        except Exception:
            break

    if len(all_data) < 50:
        return None

    df = pd.DataFrame(
        all_data,
        columns=[
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "close_time",
            "quote_asset_volume",
            "num_trades",
            "taker_buy_base_volume",
            "taker_buy_quote_volume",
            "ignore"
        ]
    )

    # ---------- Tipos ----------
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)

    # ---------- Timestamps ----------
    df["time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)

    df = df[["time", "open", "high", "low", "close", "volume"]]
    df.sort_values("time", inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.dropna(inplace=True)

    if len(df) < 50:
        return None

    return df


In [6]:
def df15_provider(symbol: str, limit: int = 120):
    """
    Provee dataframe OHLCV de 15m con indicadores técnicos
    necesarios para confirmación y monitoreo.

    Retorna:
        pd.DataFrame o None
    """

    df = get_ohlc(symbol, interval="15m", limit=limit)

    if df is None or len(df) < 50:
        return None

    df = df.sort_index()

    # ---------------- INDICADORES ---------------- #
    df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()

    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / (avg_loss + 1e-9)
    df["rsi"] = 100 - (100 / (1 + rs))

    ema_fast = df["close"].ewm(span=12, adjust=False).mean()
    ema_slow = df["close"].ewm(span=26, adjust=False).mean()

    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=9, adjust=False).mean()

    df["macd_hist"] = macd - macd_signal

    return df


In [7]:
def compute_atr_from_df(df: pd.DataFrame, period: int = 14):
    if len(df) < period + 1:
        return None

    df = df.copy()
    df["prev_close"] = df["close"].shift(1)

    tr = np.maximum.reduce([
        df["high"] - df["low"],
        (df["high"] - df["prev_close"]).abs(),
        (df["low"] - df["prev_close"]).abs()
    ])

    atr = pd.Series(tr).rolling(period).mean().iloc[-1]

    if pd.isna(atr) or atr <= 0:
        return None

    return float(atr)


In [8]:
def compute_macd_histogram(
    df: pd.DataFrame,
    fast: int = 12,
    slow: int = 26,
    signal: int = 9,
    price_col: str = "close"
) -> pd.Series:
    """
    Calcula histograma MACD clásico (MACD - Signal).

    Diseñado para:
    - Timing fino de entrada (5m)
    - Evaluar energía + aceleración
    - Sin repainting (solo usa datos cerrados)

    Retorna:
        pd.Series alineada con df.index
    """

    if df is None or len(df) < slow + signal + 5:
        return pd.Series(dtype="float64")

    price = df[price_col].astype(float)

    # =========================
    # EMAs
    # =========================
    ema_fast = price.ewm(span=fast, adjust=False).mean()
    ema_slow = price.ewm(span=slow, adjust=False).mean()

    # =========================
    # MACD
    # =========================
    macd = ema_fast - ema_slow
    signal_line = macd.ewm(span=signal, adjust=False).mean()

    # =========================
    # HISTOGRAMA
    # =========================
    macd_hist = macd - signal_line

    return macd_hist


In [9]:
def df5_provider(symbol: str, limit: int = 120):
    """
    Descarga velas de 5 minutos desde Binance (API pública),
    retorna DataFrame limpio y ordenado.

    Columnas:
        time (UTC datetime)
        open
        high
        low
        close
        volume

    Retorna:
        pd.DataFrame | None
    """

    try:
        df = get_ohlc(symbol, interval="5m", limit=limit)

        if df is None or len(df) < 50:
            return None


        # Eliminar vela abierta (NO cerrada)
        #now_utc = pd.Timestamp.utcnow().tz_localize("UTC")
        #now_utc = datetime.now(timezone.utc)
        #if df.iloc[-1]["time"] + pd.Timedelta(minutes=5) > now_utc:
        # Eliminar SIEMPRE la última vela
        df = df.iloc[:-1]

        return df

    except Exception as e:
        print(f"[WARN] df5_provider {symbol}: {e}")
        return None


In [10]:
def check_follow_through_5m(df5, direction: str):
    """
    Confirma continuación mínima en 5m.
    NO redefine reversión, solo evita entrar en pullback inmediato.
    """

    if df5 is None or len(df5) < 3:
        return False

    last = df5.iloc[-1]
    prev = df5.iloc[-2]

    if direction == "LONG":
        #print(f"{last.time}: last.close ({last.close}) > prev.open ({prev.open}), prev.close ({prev.close}), last.open ({last.open})")
        return (
            last.close > max(prev.open, prev.close) and
            last.close > last.open
        )
    else:    #SHORT
        #print(f"{last.time}: last.close ({last.close}) < prev.open ({prev.open}), prev.close ({prev.close}), last.open ({last.open})")
        return (
            last.close < min(prev.open, prev.close) and
            last.close < last.open
        )


### Simulación

In [13]:
df5 = df5_provider("YBUSDT", limit=500)
df15 = df15_provider("YBUSDT", limit=500)

ts_confirmation_formatted = datetime.strptime(ts_confirmation, format_pattern) + timedelta(hours=5)
ts_confirmation_formatted = pytz.utc.localize(ts_confirmation_formatted)


df5 = df5[df5["time"] < ts_confirmation_formatted]
df15 = df15[df15["time"] < ts_confirmation_formatted]

In [14]:
df15

,time,open,high,low,close,volume,ema_50,rsi,macd_hist
0,2026-02-28 11:00:00+00:00,0.1664,0.1671,0.1657,0.1667,220427.0,0.166700,NaN,0.000000
1,2026-02-28 11:15:00+00:00,0.1667,0.1674,0.1664,0.1673,188965.0,0.166724,NaN,0.000038
2,2026-02-28 11:30:00+00:00,0.1672,0.1688,0.1659,0.1662,401257.0,0.166703,NaN,-0.000010
3,2026-02-28 11:45:00+00:00,0.1663,0.1663,0.1643,0.1646,273088.0,0.166621,NaN,-0.000142
4,2026-02-28 12:00:00+00:00,0.1646,0.1652,0.1644,0.1651,141311.0,0.166561,NaN,-0.000185
...,...,...,...,...,...,...,...,...,...
444,2026-03-05 02:00:00+00:00,0.1699,0.1704,0.1692,0.1696,117463.0,0.170082,35.714214,-0.000414
445,2026-03-05 02:15:00+00:00,0.1696,0.1708,0.1689,0.1708,132878.0,0.170111,41.558366,-0.000407
446,2026-03-05 02:30:00+00:00,0.1708,0.1708,0.1697,0.1697,38421.0,0.170095,37.209242,-0.000459
447,2026-03-05 02:45:00+00:00,0.1697,0.1702,0.1696,0.1699,31790.0,0.170087,37.930973,-0.000461


In [ ]:
ts_confirmation_formatted

In [15]:
#symbol = "REDUSDT"
symbol = "YBUSDT"
# symbol = "EULUSDT"
direction = "LONG"
mode = "early"
#ts_confirmation = "2026-03-03T11:14:59+00:00" # HORA LOCAL RED
ts_confirmation = "2026-03-03T10:40:11+00:00" # HORA LOCAL YB
# ts_confirmation = "2026-03-04T22:09:00+00:00" # HORA LOCAL EUL

In [25]:
symbol = "YBUSDT"
direction = "LONG"
mode = "early"
ts_confirmation = "2026-03-03T10:41:11+00:00" # HORA LOCAL YB


format_pattern = "%Y-%m-%dT%H:%M:%S+00:00"
ts_confirmation_formatted = datetime.strptime(ts_confirmation, format_pattern) + timedelta(hours=5)
ts_confirmation_formatted = pytz.utc.localize(ts_confirmation_formatted)

# Parámetros por modo
if mode == "early":
    min_impulse_atr = 0.6
    max_retrace_atr = 0.35
    min_range_rel = 0.002
else:
    min_impulse_atr = 0.9
    max_retrace_atr = 0.25
    min_range_rel = 0.0035

require_follow_through = True
max_next_wick_atr = 0.5


try:
    # =============================
    # 15m DATA
    # =============================
    df15 = df15_provider(symbol, limit=500)
    df15 = df15[df15["time"] < ts_confirmation_formatted]
    print(df15)
    if df15 is None or len(df15) < 50:
        sys.exit("df15 INSUFICIENTE")

    last = df15.iloc[-1]
    prev = df15.iloc[-2]

    atr_15m = compute_atr_from_df(df15, period=14)
    if atr_15m is None or atr_15m <= 0:
        sys.exit("error atr_15m")

    candle_range = last.high - last.low
    range_rel = candle_range / last.close
    if range_rel < min_range_rel:
        print(f"\n{symbol}|range_rel: {range_rel} < {min_range_rel}")
        sys.exit("range_rel < min_range_rel")

    # =============================
    # Impulse / Retrace 15m
    # =============================
    if direction == "LONG":
        impulse = (last.close - last.low) / atr_15m
        retrace = (last.high - last.close) / atr_15m
        valid_structure = last.close > last.open
    else:
        impulse = (last.high - last.close) / atr_15m
        retrace = (last.close - last.low) / atr_15m
        valid_structure = last.close < last.open

    if impulse < min_impulse_atr:
        print(f"\n{symbol}|impulse: {impulse} < {min_impulse_atr}")
        sys.exit("impulse < min_impulse_atr")

    if retrace > max_retrace_atr:
        print(f"{symbol}|retrace: {retrace} > {max_retrace_atr}")
        sys.exit("retrace > max_retrace_atr")

    if not valid_structure:
        print(f"{symbol}|valid_structure: {valid_structure}")
        sys.exit("NOT valid_structure")

    # =============================
    # Barrida de liquidez 15m
    # =============================
    if direction == "LONG":
        adverse_wick = (last.low - prev.close) / atr_15m
    else:
        adverse_wick = (prev.close - last.high) / atr_15m

    if adverse_wick < -max_next_wick_atr:
        print(f"{symbol}|adverse_wick: {adverse_wick} < {-max_next_wick_atr}")
        sys.exit("adverse_wick < -max_next_wick_atr")

    # =============================
    # Follow Through 5m
    # =============================
    if require_follow_through:
        df5 = df5_provider(symbol, limit=500)
        df5 = df5[df5["time"] < ts_confirmation_formatted]
        if df5 is None or len(df5) < 3:
            sys.exit("df5 INSUFICIENTE")

        if not check_follow_through_5m(df5, direction):
            print(f"{symbol}|check_follow_through_5m: FALSE")
            sys.exit("NOT check_follow_through_5m")

    # =============================
    # MACD ENERGY 5m
    # =============================
    df5 = df5_provider(symbol, limit=500)
    df5 = df5[df5["time"] < ts_confirmation_formatted]
    print(df5)
    if df5 is None or len(df5) < 120:
        sys.exit("df5 INSUFICIENTE")

    macd_hist = compute_macd_histogram(df5)

    if macd_hist is None or len(macd_hist) < 100:
        sys.exit("macd_hist INSUFICIENTE")

    h0 = macd_hist.iloc[-2]
    h1 = macd_hist.iloc[-3]
    h2 = macd_hist.iloc[-4]

    abs_hist = abs(macd_hist.iloc[-100:])

    # 1️⃣ Dirección correcta
    if direction == "LONG":
        direction_ok = h0 > 0
    else:
        direction_ok = h0 < 0
    
    if not direction_ok:
        print(f"{symbol}|direction_ok: {direction_ok}")
        sys.exit("NOT direction_ok")

    # 2️⃣ Aceleración obligatoria
    accelerating = abs(h0) > abs(h1) > abs(h2)
    print(f"h0:{abs(h0)} > h1:{abs(h1)} > h2:{abs(h2)}")
    if not accelerating:
        print(f"{symbol}|accelerating: {accelerating}")
        sys.exit("NOT accelerating")

    
    # 3️⃣ Convexidad creciente
    delta1 = abs(h0) - abs(h1)
    delta2 = abs(h1) - abs(h2)
    if delta1 <= delta2:
        sys.exit("Convexidad decreciente")
    
    # Si está acelerando fuerte → percentil más permisivo
    # momentum_strength = abs(h0 - h1)
    
    recent_volatility = abs_hist.std()
    
    if recent_volatility == 0:
        print(f"{symbol}|recent_volatility: {recent_volatility}")
        sys.exit("recent_volatility = 0")
    
    acceleration_ratio = (abs(h0) - abs(h1)) / abs(h1)

    # 3️⃣ Percentil dinámico adaptativo
    if acceleration_ratio > 1.0:
        base_percentile = 78
    elif acceleration_ratio > 0.7:
        base_percentile = 82
    elif acceleration_ratio > 0.4:
        base_percentile = 85
    elif acceleration_ratio >= 0.25:
        base_percentile = 88
    else:
        sys.exit("acceleration_ratio < 0.25")
    
    threshold = np.percentile(abs_hist, base_percentile)
    
    macd_energy = abs(h0)
    print(f"{symbol}|macd_energy: {macd_energy} > {threshold} (acceleration_ratio:{acceleration_ratio})")
    if macd_energy < threshold:
        sys.exit("macd_energy < threshold")

    # 4️⃣ Protección contra sobreextensión
    upper_extreme = np.percentile(abs_hist, 92)
    print(f"{symbol}|macd_energy: {macd_energy} < {upper_extreme}")
    if macd_energy > upper_extreme:
        # Movimiento demasiado extendido → probable agotamiento
        sys.exit("macd_energy > upper_extreme")

    # =============================
    # CONFIRMACIÓN FINAL
    # =============================
    msg = (
        f"🟢 ENTRADA CONFIRMADA\n"
        f"{symbol} {direction} | {datetime.now()}\n"
        f"Modo: {mode}\n"
        f"Precio: {round(last.close, 6)}\n"
        f"ATR15m: {round(atr_15m, 6)}\n"
        f"Impulse: {round(impulse, 2)} ATR\n"
        f"Retrace: {round(retrace, 2)} ATR"
    )

    send_telegram_alert(msg)
    print(msg)
    local_warning()

except Exception as e:
    print(f"[WARN] {symbol}: {e}")


                         time    open    high     low   close    volume  \
0   2026-02-28 12:15:00+00:00  0.1650  0.1652  0.1646  0.1649   87978.0   
1   2026-02-28 12:30:00+00:00  0.1649  0.1650  0.1627  0.1629  353299.0   
2   2026-02-28 12:45:00+00:00  0.1628  0.1630  0.1620  0.1628  217685.0   
3   2026-02-28 13:00:00+00:00  0.1628  0.1633  0.1627  0.1629  100704.0   
4   2026-02-28 13:15:00+00:00  0.1630  0.1634  0.1590  0.1595  855752.0   
..                        ...     ...     ...     ...     ...       ...   
297 2026-03-03 14:30:00+00:00  0.1676  0.1685  0.1658  0.1659  292505.0   
298 2026-03-03 14:45:00+00:00  0.1658  0.1662  0.1642  0.1654  458443.0   
299 2026-03-03 15:00:00+00:00  0.1655  0.1666  0.1655  0.1663  176980.0   
300 2026-03-03 15:15:00+00:00  0.1663  0.1667  0.1649  0.1665   99731.0   
301 2026-03-03 15:30:00+00:00  0.1664  0.1682  0.1664  0.1675  101649.0   

       ema_50        rsi  macd_hist  
0    0.164900        NaN   0.000000  
1    0.164822        Na

SystemExit: retrace > max_retrace_atr

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Resultados

In [45]:
print(f"h0 = {h0}")
print(f"h1 = {h1}")
print(f"h2 = {h2}")
print(f"acceleration_ratio = {acceleration_ratio}")
print(f"base_percentile = {base_percentile}")
print(f"threshold = {threshold}")
print(f"upper_extreme = {upper_extreme}")
print(f"macd_energy = {macd_energy}")
print(f"direction_ok = {direction_ok}")
print(f"accelerating = {accelerating}")
print(f"passes_threshold = {macd_energy > threshold}") 
print(f"passes_overextension = {macd_energy < upper_extreme}")

NameError: name 'h0' is not defined